# Results Plots Script

## Initializations

### Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### Constants

In [ ]:
BASE_PATH = "./results/london_house_price"
ALGOS = ["datafly", "incognito", "oka", "kmember", "classic_mondrian"]
K = [2] + list(range(5, 101, 5))
MARKERS = {
    "incognito": "$i$",
    "oka": "$o$",
    "kmember": "$k$",
    "classic_mondrian": "$m$",
    "datafly": "$d$",
}
COLORS = {
    "sample": "tab:green",
    "incognito": "tab:olive",
    "oka": "tab:blue",
    "kmember": "tab:red",
    "classic_mondrian": "tab:purple",
    "datafly": "tab:brown",
}

## Plots

### Data Utility

In [ ]:
def get_ut_score(results_df):
    return results_df.set_index("metric").T.to_dict(orient="records")[0]


def get_all_ut_scores(path, all_k):
    all_ut_scores = []
    for k in all_k:
        all_ut_scores.append(get_ut_score(pd.read_csv(f"{path}/k_{k}/results_ut.csv")))
    return all_ut_scores

In [ ]:
# sample_ut_scores = get_ut_score(pd.read_csv(f"{BASE_PATH}/sample/results_ut.csv"))
all_algos_ut_scores = {}
for algo in ALGOS:
    all_algos_ut_scores[algo] = get_all_ut_scores(f"{BASE_PATH}/{algo}", K)

In [ ]:
sample_ut_scores = {'IL': 0}

#### $NCP$

In [ ]:
fig, axis = plt.subplots(1, 1, figsize=[6, 4])

baseline = [0] * len([0] + K + [105])
axis.plot(
    [0] + K + [105],
    baseline,
    label="best effort",
    color=COLORS["sample"],
)

for algo in list(all_algos_ut_scores):
    axis.plot(
        K,
        [x["IL"] for x in all_algos_ut_scores[algo]],
        label=algo,
        color=COLORS[algo],
        marker=MARKERS[algo],
        linestyle=":",
    )

axis.legend(
    ncol=len(ALGOS) / 2,
    fontsize="medium",
    bbox_to_anchor=(0.32, 0.05),
)
axis.set_ylim([-0.02, 1.02])
axis.set_ylabel("NCP")
axis.set_xlabel("k")
axis.set_xlim([0, 105])
axis.set_xticks(K[1:])
axis.set_yticks(np.arange(0, 1.05, 0.1))
axis.grid(linestyle=":")
fig.tight_layout(h_pad=0)

fig.savefig(f"./figs/result_adult_ut_NCP.png", dpi=150, bbox_inches='tight', pad_inches=0.015)

### ML Classification Results

In [ ]:
def get_f1_score(results_df):
    return (
        results_df[["model", "f1_score"]]
        .set_index("model")
        .T.to_dict(orient="records")[0]
    )


def get_all_validation_f1_scores(path, all_k):
    all_f1_scores = []
    for k in all_k:
        all_f1_scores.append(
            get_f1_score(pd.read_csv(f"{path}/k_{k}/results_ml_validation.csv"))
        )
    return all_f1_scores

In [ ]:
all_algos_validation_f1_scores = {}
for algo in ALGOS:
    all_algos_validation_f1_scores[algo] = get_all_validation_f1_scores(
        f"{BASE_PATH}/{algo}", K
    )

In [ ]:
sample_f1_scores = get_f1_score(pd.read_csv(f"{BASE_PATH}/sample/results_ml_validation.csv"))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=[15, 4])
codes = ["(a)", "(b)", "(c)"]
models = [x for x in list(sample_f1_scores) if x != "SVM"]

for idx, model in enumerate(models):
    sample_scores = [sample_f1_scores[model]] * len(K)
    axis = ax[idx]
    axis.plot(
        K,
        sample_scores,
        label="sample",
        color=COLORS["sample"],
    )
    for algo in all_algos_validation_f1_scores:
        axis.plot(
            K,
            [x[model] for x in all_algos_validation_f1_scores[algo]],
            label=algo,
            color=COLORS[algo],
            marker=MARKERS[algo],
            linestyle=":",
        )

    axis.legend(
        ncol=len(ALGOS) / 2,
        loc="lower right",
        fontsize="medium",
    )
    axis.set_title(f"{codes[idx]} {model}", y=-0.25, fontsize=15)
    axis.set_ylim([0, 1])
    axis.set_ylabel("F1 Score")
    axis.set_xlabel(f"k")
    axis.set_xlim([0, 105])
    axis.set_xticks(K[1:])
    axis.set_yticks(np.arange(0.0, 1.1, 0.1))
    axis.grid(linestyle=":")
fig.tight_layout(h_pad=1.8)
fig.savefig(
    f"./figs/result_adult_ml.png",
    dpi=150,
    bbox_inches="tight",
    pad_inches=0.015,
)

### Vulnerability

In [ ]:
def get_attack_result(path):
    return pd.read_csv(path)["correct"].sum().item()

In [ ]:
all_attack_results = {}
for algo in ALGOS:
    all_attack_results[algo] = [
        pd.read_csv(f"{BASE_PATH}/{algo}/k_{k}/results_attack.csv")["correct"]
        .sum()
        .item()
        for k in K
    ]

In [ ]:
fig, (baseline_axis, axis) = plt.subplots(
    2, 1, figsize=[6, 4], sharex=True, gridspec_kw={"height_ratios": [1, 9]}
)

axis.plot(
    range(0, 106, 1),
    [float("nan")] * 106,
    label="max (worst)",
    color=COLORS["sample"],
)

for algo in list(all_attack_results):
    axis.plot(
        K,
        np.array(all_attack_results[algo]) / 3000,
        label=algo,
        color=COLORS[algo],
        marker=MARKERS[algo],
        linestyle=":",
    )

axis.legend(
    ncol=len(ALGOS) / 2,
    fontsize="medium",
)
axis.set_ylim([0, 0.55])
axis.set_ylabel("Match Ratio")
axis.set_xlabel("k")
axis.set_xlim([0, 105])
axis.set_xticks(K[1:])
axis.set_yticks(np.arange(0, 0.56, 0.05))

baseline = [1] * 106
baseline_axis.plot(
    range(0, 106, 1),
    baseline,
    label="best effort",
    color=COLORS["sample"],
)

baseline_axis.set_ylim([0.88, 1.1])
baseline_axis.set_yticks([1])
baseline_axis.spines.bottom.set_visible(False)
baseline_axis.tick_params(bottom=False, labeltop=False)
axis.xaxis.tick_bottom()
axis.spines.top.set_visible(False)

d = -0.5  # proportion of vertical to horizontal extent of the slanted line
kwargs = dict(
    marker=[(-1, -d), (1, d)],
    markersize=8,
    linestyle="--",
    alpha=0.3,
    color="k",
    mec="k",
    mew=1,
    clip_on=False,
)
axis.plot([0, 1], [1, 1], transform=axis.transAxes, **kwargs)
baseline_axis.plot([0, 1], [0, 0], transform=baseline_axis.transAxes, **kwargs)
axis.grid(linestyle=":")
baseline_axis.grid(linestyle=":")
fig.tight_layout(h_pad=0)

fig.savefig(
    f"./figs/result_adult_vulnerability.png", dpi=150, bbox_inches="tight", pad_inches=0
)